In [ ]:
import warnings
from pathlib import Path
 
import sys
import warnings
from pathlib import Path
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
 
warnings.filterwarnings("ignore")
out = Path(sys.argv[2] if len(sys.argv) > 2 else "output")
(out / "charts").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
RED, GREY, BLUE = "#E50914", "#564d4d", "#2b6cb0"

In [3]:
df = pd.read_parquet('../prepared_data.parquet')
df["type"] = df["type"].astype(str)
last_added = df["date_added"].max()
PARTIAL_YEAR = int(last_added.year)          # data stops mid-year -> incomplete
counts = df["release_year"].value_counts().sort_index()
counts = counts.reindex(range(counts.index.min(), counts.index.max() + 1), fill_value=0)
 
START = 2000                                  # earlier years are tiny and sparse
full = counts.loc[START:PARTIAL_YEAR - 1].astype(float)   # complete years only
partial_obs = counts.loc[PARTIAL_YEAR]
print(f"Data ends {last_added.date()}. Training on {START}-{PARTIAL_YEAR - 1}; "
      f"{PARTIAL_YEAR} is partial ({int(partial_obs)} titles so far).")

Data ends 2021-09-25. Training on 2000-2020; 2021 is partial (219 titles so far).


In [4]:
yoy = full.pct_change() * 100
peak_year = int(full.idxmax())
trend = pd.DataFrame({"titles": full.astype(int), "yoy_%": yoy.round(1),
                      "3yr_avg": full.rolling(3).mean().round(1)})
trend.to_csv(out / "yearly_trend.csv")
print(trend.tail(10))
cagr = (full.loc[peak_year] / full.loc[START]) ** (1 / (peak_year - START)) - 1
print(f"Peak: {peak_year} ({int(full.max())} titles). CAGR {START}->{peak_year}: {cagr:.1%}")
post = (full.iloc[-1] / full.loc[peak_year]) ** (1 / (full.index[-1] - peak_year)) - 1
print(f"Change since peak ({peak_year}->{full.index[-1]}): {post:.1%} per year")

              titles  yoy_%  3yr_avg
release_year                        
2011             144   -2.7    137.0
2012             175   21.5    155.7
2013             223   27.4    180.7
2014             262   17.5    220.0
2015             386   47.3    290.3
2016             645   67.1    431.0
2017             744   15.3    591.7
2018             730   -1.9    706.3
2019             598  -18.1    690.7
2020             503  -15.9    610.3
Peak: 2017 (744 titles). CAGR 2000->2017: 20.1%
Change since peak (2017->2020): -12.2% per year


In [5]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(full.index, full.values, color=RED, label="Titles by release year")
ax.bar([PARTIAL_YEAR], [partial_obs], color=RED, alpha=0.35, hatch="//",
       label=f"{PARTIAL_YEAR} (partial, to {last_added:%b})")
ax.plot(full.index, full.rolling(3).mean(), color=GREY, lw=2, label="3-year average")
ax.axvline(peak_year, color="black", ls=":", lw=1)
ax.text(peak_year + 0.2, full.max() * 0.97, f"peak {peak_year}", fontsize=9)
ax.set(title="Netflix titles by release year", xlabel="Release year", ylabel="Titles")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig(out / "charts/01_yearly_trend.png", dpi=150); plt.close()

In [6]:
fig, ax = plt.subplots(figsize=(10, 4))
cols = [RED if v >= 0 else GREY for v in yoy.dropna()]
ax.bar(yoy.dropna().index, yoy.dropna().values, color=cols)
ax.axhline(0, color="black", lw=0.8)
ax.set(title="Year-over-year growth in titles released", xlabel="Release year", ylabel="% change")
plt.tight_layout(); plt.savefig(out / "charts/02_yoy_growth.png", dpi=150); plt.close()

In [7]:
by_type = (df[df["release_year"].between(START, PARTIAL_YEAR)]
           .groupby(["release_year", "type"]).size().unstack(fill_value=0))
fig, ax = plt.subplots(figsize=(10, 4.2))
by_type.plot(ax=ax, color=[RED, GREY], lw=2.2)
ax.set(title="Movies vs TV shows by release year", xlabel="Release year", ylabel="Titles")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(out / "charts/03_movies_vs_tv.png", dpi=150); plt.close()

In [8]:
def _band(mean, resid_sd, h, z=1.2816):          # 80% interval
    sd = resid_sd * np.sqrt(np.arange(1, h + 1))
    return mean - z * sd, mean + z * sd
 
 
def f_naive(y, h):
    sd = y.diff().dropna().std()
    m = np.repeat(y.iloc[-1], h)
    return (m, *_band(m, sd, h))
 
 
def f_ma3(y, h):
    m = np.repeat(y.iloc[-3:].mean(), h)
    sd = (y - y.rolling(3).mean().shift(1)).dropna().std()
    return (m, *_band(m, sd, h))
 
 
def _ols(y, h, log, window=10):
    w = y.iloc[-window:]
    X = sm.add_constant(np.arange(len(w)))
    yy = np.log(w.values) if log else w.values
    res = sm.OLS(yy, X).fit()
    Xf = sm.add_constant(np.arange(len(w), len(w) + h), has_constant="add")
    pf = res.get_prediction(Xf).summary_frame(alpha=0.2)
    m, lo, hi = pf["mean"].values, pf["obs_ci_lower"].values, pf["obs_ci_upper"].values
    if log:
        m, lo, hi = np.exp(m), np.exp(lo), np.exp(hi)
    return np.maximum(m, 0), np.maximum(lo, 0), np.maximum(hi, 0)

In [9]:
def f_linear(y, h):  return _ols(y, h, False)
def f_loglinear(y, h): return _ols(y, h, True)
 
 
def _ets(y, h, damped):
    ys = pd.Series(y.values, index=pd.RangeIndex(len(y)))   # ETS needs an indexed Series
    res = ETSModel(ys, error="add", trend="add", damped_trend=damped).fit(disp=False)
    pf = res.get_prediction(start=len(y), end=len(y) + h - 1).summary_frame(alpha=0.2)
    return (np.maximum(pf["mean"].values, 0), np.maximum(pf["pi_lower"].values, 0),
            np.maximum(pf["pi_upper"].values, 0))
 
 
def f_holt(y, h):   return _ets(y, h, False)
def f_damped(y, h): return _ets(y, h, True)
 
 
def f_arima(y, h):
    res = ARIMA(y.values, order=(1, 1, 1)).fit()
    fc = res.get_forecast(h)
    ci = fc.conf_int(alpha=0.2)
    return np.maximum(fc.predicted_mean, 0), np.maximum(ci[:, 0], 0), np.maximum(ci[:, 1], 0)
 
 
MODELS = {
    "Naive (last value)": f_naive,
    "3-yr moving average": f_ma3,
    "Linear trend (10 yr)": f_linear,
    "Log-linear trend (10 yr)": f_loglinear,
    "Holt (additive trend)": f_holt,
    "Damped Holt": f_damped,
    "ARIMA(1,1,1)": f_arima,
}

In [10]:
H = 3
origins = range(2012, full.index[-1] - 0)       # last origin leaves >=1 actual year
rows = []
for name, fn in MODELS.items():
    for t in origins:
        train = full.loc[:t]
        n_ahead = min(H, int(full.index[-1] - t))
        if n_ahead < 1:
            continue
        try:
            pred = fn(train, n_ahead)[0]
        except Exception as e:                    # never hide a failing model
            print(f"  [skip] {name} origin {t}: {type(e).__name__}: {e}")
            continue
        actual = full.loc[t + 1:t + n_ahead].values
        for k in range(n_ahead):
            rows.append({"model": name, "origin": t, "h": k + 1,
                         "pred": pred[k], "actual": actual[k]})
bt = pd.DataFrame(rows)
bt["err"] = bt["pred"] - bt["actual"]
bt["ape"] = (bt["err"].abs() / bt["actual"]) * 100
 
 
def summarize(g):
    return pd.Series({"MAE": g["err"].abs().mean(), "RMSE": np.sqrt((g["err"] ** 2).mean()),
                      "MAPE_%": g["ape"].mean(), "bias": g["err"].mean(), "n": len(g)})

In [11]:
score = bt.groupby("model").apply(summarize).sort_values("RMSE").round(1)
print("\nBacktest (all horizons 1-3, origins 2012+):\n", score)
score.to_csv(out / "model_scores.csv")
score_h1 = bt[bt.h == 1].groupby("model").apply(summarize).sort_values("RMSE").round(1)
score_h3 = bt[bt.h == 3].groupby("model").apply(summarize).sort_values("RMSE").round(1)
score_h1.to_csv(out / "model_scores_h1.csv"); score_h3.to_csv(out / "model_scores_h3.csv")
 
best = score.index[0]
print("Best model by RMSE:", best)
 
cmp_ = pd.DataFrame({"1 year ahead": score_h1["RMSE"], "3 years ahead": score_h3["RMSE"],
                     "All horizons": score["RMSE"]}).loc[score.index[::-1]]
ax = cmp_.plot(kind="barh", figsize=(9.5, 5.2), color=[RED, GREY, "#c9c4c4"], width=0.8)
ax.set(title="Backtest error by model (RMSE, titles per year; lower is better)", xlabel="RMSE", ylabel="")
ax.legend(title="Forecast horizon", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout(); plt.savefig(out / "charts/04_model_comparison.png", dpi=150); plt.close()


Backtest (all horizons 1-3, origins 2012+):
                             MAE   RMSE  MAPE_%   bias     n
model                                                      
Naive (last value)        190.8  233.5    34.1 -104.9  21.0
Linear trend (10 yr)      223.0  259.5    39.4 -100.6  21.0
3-yr moving average       239.5  282.1    42.2 -183.5  21.0
ARIMA(1,1,1)              230.3  296.6    40.0   25.7  21.0
Damped Holt               232.7  306.0    40.7   40.7  21.0
Holt (additive trend)     249.1  322.9    43.7   56.9  21.0
Log-linear trend (10 yr)  258.4  333.2    46.1   36.1  21.0
Best model by RMSE: Naive (last value)


In [12]:
FH = 5
future_years = np.arange(full.index[-1] + 1, full.index[-1] + 1 + FH)
fc_rows = {}
best_h1 = score_h1.index[0]
print("Best 1-year-ahead model:", best_h1)
for name in dict.fromkeys([best, best_h1, "Damped Holt", "ARIMA(1,1,1)", "Linear trend (10 yr)"]):
    m, lo, hi = MODELS[name](full, FH)
    fc_rows[name] = pd.DataFrame({"year": future_years, "forecast": m.round(0),
                                  "low_80": lo.round(0), "high_80": hi.round(0)})
fc = fc_rows[best]
fc.to_csv(out / "forecast_best_model.csv", index=False)
pd.concat({k: v.set_index("year") for k, v in fc_rows.items()}).to_csv(out / "forecast_all_models.csv")
print(f"\nForecast ({best}):\n", fc)

Best 1-year-ahead model: ARIMA(1,1,1)

Forecast (Naive (last value)):
    year  forecast  low_80  high_80
0  2021     503.0   403.0    603.0
1  2022     503.0   362.0    644.0
2  2023     503.0   330.0    676.0
3  2024     503.0   303.0    703.0
4  2025     503.0   279.0    727.0


In [14]:
fig, ax = plt.subplots(figsize=(10, 5))
hist = full.loc[2005:]
ax.plot(hist.index, hist.values, color="black", lw=2, marker="o", ms=3, label="Actual (complete years)")
ax.scatter([PARTIAL_YEAR], [partial_obs], color="black", facecolors="none", s=60, zorder=5,
           label=f"{PARTIAL_YEAR} so far (to {last_added:%b}; incomplete)")
ax.plot(fc["year"], fc["forecast"], color=RED, lw=2.5, marker="o", label=f"Forecast: {best}")
ax.fill_between(fc["year"], fc["low_80"], fc["high_80"], color=RED, alpha=0.18, label="80% interval")
for alt, c in [(best_h1, BLUE), ("Linear trend (10 yr)", GREY)]:
    if alt != best:
        a = fc_rows[alt]
        tag = "best 1-yr-ahead" if alt == best_h1 else "for comparison"
        ax.fill_between(a["year"], a["low_80"], a["high_80"], color=c, alpha=0.10)
        ax.plot(a["year"], a["forecast"], color=c, lw=1.8, ls="--", marker="s", ms=3,
                label=f"{alt} ({tag})")
ax.axvline(full.index[-1] + 0.5, color="grey", ls=":", lw=1)
ax.set(title=f"Titles by release year - forecast {future_years[0]}-{future_years[-1]}",
       xlabel="Release year", ylabel="Titles")
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig(out / "charts/05_forecast.png", dpi=150); plt.close()

In [15]:
t0 = 2016
train = full.loc[:t0]
m, lo, hi = MODELS[best](train, full.index[-1] - t0)
yrs = np.arange(t0 + 1, full.index[-1] + 1)
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(full.loc[2008:].index, full.loc[2008:].values, color="black", lw=2, label="Actual")
ax.plot(yrs, m, color=RED, lw=2.2, marker="o", label=f"{best} trained to {t0}")
ax.fill_between(yrs, lo, hi, color=RED, alpha=0.18)
ax.axvline(t0 + 0.5, color="grey", ls=":")
ax.set(title=f"Backtest: forecast made from data up to {t0}", xlabel="Release year", ylabel="Titles")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()
plt.tight_layout(); plt.savefig(out / "charts/06_backtest_example.png", dpi=150); plt.close()
 
print("\nDone. Outputs in", out.resolve())


Done. Outputs in D:\Python\netflix-recommendation-system-and-trend-projection\task 4\output
